# Notebook 5: Results & Visualizations\n**Project:** Can FDA Drug Approvals Predict Stock Price Movements?  \n**Author:** Hari Vykuntapu | MS Artificial Intelligence, Southwest Baptist University  \n\n---\n\nThe numbers are in. XGBoost wins on AUROC at 0.5725 — above random, but not by a margin you'd retire on. That's an honest result for 296 samples and a 7-day prediction window with a lot of market noise baked in.\n\nThe part I didn't expect: FinBERT ranked as the top SHAP feature, above my RSS score. I built this project assuming that structured regulatory metadata would outperform NLP sentiment — that the classification codes would carry more signal than anything a transformer could extract from formulaic regulatory prose. The data partially confirms that, but it's more nuanced. FinBERT is doing real work, and RSS is doing real work, and they're not measuring the same thing. That's actually the more interesting finding.\n\nThe charts below break down what drove those predictions and whether the patterns make pharmaceutical market sense.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for execution
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split

try:
    import shap
    SHAP_AVAILABLE = True
    print('SHAP available.')
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed — using built-in feature importances.')

os.makedirs('../outputs/results', exist_ok=True)
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False
})

print('Setup complete.')

SHAP available.
Setup complete.


In [2]:
df = pd.read_csv('../data/processed/fda_features.csv')
metrics_df = pd.read_csv('../outputs/results/model_metrics.csv', index_col=0)

with open('../models/model_metadata.json') as f:
    metadata = json.load(f)

FEATURE_COLS = [c for c in metadata['feature_cols'] if c in df.columns]
TARGET = metadata['target']
RANDOM_STATE = 42

X = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

best_model = joblib.load('../models/best_model.pkl')

lr_model, rf_model, xgb_model = None, None, None
for fname, attr in [('logistic_regression.pkl', 'lr_model'),
                    ('random_forest.pkl', 'rf_model'),
                    ('xgboost.pkl', 'xgb_model')]:
    path = f'../models/{fname}'
    if os.path.exists(path):
        try:
            locals()[attr]  # just to reference the variable
        except:
            pass
        try:
            if attr == 'lr_model': lr_model = joblib.load(path)
            elif attr == 'rf_model': rf_model = joblib.load(path)
            elif attr == 'xgb_model': xgb_model = joblib.load(path)
        except Exception as e:
            print(f'Could not load {fname}: {e}')

print(f'Best model: {metadata["best_model"]}')
print(metrics_df.round(4))

Best model: XGBoost
                     accuracy  precision  recall      f1   auroc
model                                                           
Logistic Regression    0.5333     0.5455  0.7500  0.6316  0.4431
Random Forest          0.5000     0.5263  0.6250  0.5714  0.5525
XGBoost                0.5333     0.5625  0.5625  0.5625  0.5725


## Plot 1: Model Comparison Bar Chart

In [3]:
metric_cols = [c for c in ['accuracy', 'precision', 'recall', 'f1', 'auroc'] if c in metrics_df.columns]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(metric_cols))
n_models = len(metrics_df)
width = 0.8 / max(n_models, 1)
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

for i, (model_name, row) in enumerate(metrics_df.iterrows()):
    vals = [float(row[c]) for c in metric_cols]
    offset = (i - n_models / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width=width * 0.9,
                  label=model_name, color=colors[i % len(colors)], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metric_cols])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold', pad=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.axhline(0.5, color='#888', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/results/01_model_comparison.png', bbox_inches='tight')
plt.close()
print('Saved: 01_model_comparison.png')

Saved: 01_model_comparison.png


## Plot 2: ROC Curves

In [4]:
fig, ax = plt.subplots(figsize=(7, 6))
models_to_plot = []
if lr_model: models_to_plot.append(('Logistic Regression', lr_model, '#2196F3'))
if rf_model: models_to_plot.append(('Random Forest', rf_model, '#4CAF50'))
if xgb_model: models_to_plot.append(('XGBoost', xgb_model, '#FF5722'))

# Always include best model if not already covered
if not models_to_plot:
    models_to_plot.append((metadata['best_model'], best_model, '#2196F3'))

for name, model, color in models_to_plot:
    try:
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', color=color, linewidth=2)
    except Exception as e:
        print(f'ROC skipped for {name}: {e}')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — FDA Approval Stock Prediction', fontweight='bold', pad=12)
ax.legend(loc='lower right')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig('../outputs/results/02_roc_curves.png', bbox_inches='tight')
plt.close()
print('Saved: 02_roc_curves.png')

Saved: 02_roc_curves.png


## Plot 3: Confusion Matrix

In [5]:
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Stock DOWN', 'Stock UP']
).plot(ax=ax, colorbar=True, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix — {metadata["best_model"]}', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../outputs/results/03_confusion_matrix.png', bbox_inches='tight')
plt.close()
print('Saved: 03_confusion_matrix.png')

Saved: 03_confusion_matrix.png


## Plot 4: Feature Importance (SHAP or Built-in)

In [6]:
def plot_builtin_importance(model, feature_names, save_path):
    fig, ax = plt.subplots(figsize=(9, 5))
    importances = None

    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'named_steps'):
        clf = model.named_steps.get('clf')
        if clf and hasattr(clf, 'coef_'):
            importances = np.abs(clf.coef_[0])
        elif clf and hasattr(clf, 'feature_importances_'):
            importances = clf.feature_importances_

    if importances is None:
        importances = np.ones(len(feature_names))

    idx = np.argsort(importances)
    names = [feature_names[i] for i in idx]
    vals = importances[idx]
    bar_colors = ['#FF5722' if 'rss' in n else '#2196F3' for n in names]

    ax.barh(names, vals, color=bar_colors, alpha=0.85)
    ax.set_xlabel('Feature Importance')
    ax.set_title('Feature Importance (Built-in)', fontweight='bold', pad=12)

    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#FF5722', label='RSS-derived'),
                       Patch(color='#2196F3', label='Other')], loc='lower right')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    return importances


shap_done = False
if SHAP_AVAILABLE:
    try:
        if hasattr(best_model, 'named_steps'):
            clf_shap = best_model.named_steps['clf']
            X_shap = best_model.named_steps['scaler'].transform(X_test)
        else:
            clf_shap = best_model
            X_shap = X_test.values

        if hasattr(clf_shap, 'estimators_') or hasattr(clf_shap, 'get_booster'):
            explainer = shap.TreeExplainer(clf_shap)
        else:
            bg = shap.sample(pd.DataFrame(X_shap, columns=FEATURE_COLS), min(30, len(X_shap)))
            explainer = shap.KernelExplainer(clf_shap.predict_proba, bg)

        shap_vals = explainer.shap_values(X_shap)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]

        # Beeswarm
        plt.figure(figsize=(9, 6))
        shap.summary_plot(shap_vals, X_shap, feature_names=FEATURE_COLS,
                          plot_type='dot', show=False, max_display=10)
        plt.title('SHAP Feature Contributions', fontweight='bold')
        plt.tight_layout()
        plt.savefig('../outputs/results/04_shap_beeswarm.png', bbox_inches='tight')
        plt.close()
        print('Saved: 04_shap_beeswarm.png')

        # Bar
        plt.figure(figsize=(9, 5))
        shap.summary_plot(shap_vals, X_shap, feature_names=FEATURE_COLS,
                          plot_type='bar', show=False, max_display=10)
        plt.title('SHAP Mean Absolute Feature Importance', fontweight='bold')
        plt.tight_layout()
        plt.savefig('../outputs/results/04b_shap_bar.png', bbox_inches='tight')
        plt.close()
        print('Saved: 04b_shap_bar.png')
        shap_done = True

    except Exception as e:
        print(f'SHAP analysis failed: {e}. Falling back to built-in importances.')

if not shap_done:
    plot_builtin_importance(best_model, FEATURE_COLS, '../outputs/results/04_feature_importance.png')
    print('Saved: 04_feature_importance.png')

Saved: 04_shap_beeswarm.png
Saved: 04b_shap_bar.png


## Plot 5: RSS Score vs Stock Movement

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

up_mask = df['price_up_7d'] == 1
down_mask = df['price_up_7d'] == 0

# Scatter with trend line
axes[0].scatter(df.loc[down_mask, 'rss_score'], df.loc[down_mask, 'return_7d'],
                alpha=0.45, color='#F44336', s=25, label='Stock DOWN')
axes[0].scatter(df.loc[up_mask, 'rss_score'], df.loc[up_mask, 'return_7d'],
                alpha=0.45, color='#4CAF50', s=25, label='Stock UP')

valid = df[['rss_score', 'return_7d']].dropna()
if len(valid) > 2 and valid['rss_score'].std() > 0:
    z = np.polyfit(valid['rss_score'], valid['return_7d'], 1)
    x_line = np.linspace(valid['rss_score'].min(), valid['rss_score'].max(), 100)
    axes[0].plot(x_line, np.poly1d(z)(x_line), 'k--', linewidth=1.5, alpha=0.7, label='Trend')

axes[0].axhline(0, color='#888', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].set_xlabel('RSS Score'); axes[0].set_ylabel('7-Day % Return')
axes[0].set_title('RSS Score vs 7-Day Stock Return', fontweight='bold')
axes[0].legend()

# Boxplot
up_vals = df.loc[up_mask, 'rss_score'].dropna()
down_vals = df.loc[down_mask, 'rss_score'].dropna()

bp = axes[1].boxplot([down_vals, up_vals], labels=['DOWN', 'UP'],
                      patch_artist=True, medianprops={'color': 'black', 'linewidth': 2})
bp['boxes'][0].set_facecolor('#F44336'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#4CAF50'); bp['boxes'][1].set_alpha(0.6)
axes[1].set_title('RSS Distribution: UP vs DOWN', fontweight='bold')
axes[1].set_ylabel('RSS Score'); axes[1].set_xlabel('7-Day Outcome')

for i, vals in enumerate([down_vals, up_vals], 1):
    if len(vals) > 0:
        axes[1].text(i, vals.max() + 0.005, f'μ={vals.mean():.3f}',
                     ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/results/05_rss_vs_movement.png', bbox_inches='tight')
plt.close()
print('Saved: 05_rss_vs_movement.png')

Saved: 05_rss_vs_movement.png


## Plot 6: FinBERT vs VADER Sentiment Comparison

This comparison matters beyond just which one scores better. VADER tends to skew positive on FDA approval text because words like "approval" and "accepted" have positive valence in general English. FinBERT was trained on financial news — it should in theory be more calibrated. But FDA regulatory prose is a different register from news articles too, so neither model is perfectly adapted. What I want to see: do they agree on the same events, or are they capturing different aspects of the text?

In [8]:
def safe_distribution_plot(series, ax, color, label, bins=25):
    """Try KDE; fall back to histogram if KDE fails (e.g. near-zero variance)."""
    s = series.dropna()
    if len(s) < 5:
        return
    try:
        if s.std() < 1e-6:
            raise ValueError('Near-zero variance — KDE not meaningful')
        s.plot.kde(ax=ax, color=color, linewidth=2, label=label)
    except Exception:
        ax.hist(s, bins=bins, color=color, alpha=0.5, label=label, density=True)


if 'finbert_compound' in df.columns and 'vader_compound' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Scatter: FinBERT vs VADER
    sc = axes[0].scatter(
        df['vader_compound'], df['finbert_compound'],
        alpha=0.4, c=df['price_up_7d'].astype(float), cmap='RdYlGn', s=25
    )
    axes[0].set_xlabel('VADER Compound'); axes[0].set_ylabel('FinBERT Compound')
    axes[0].set_title('FinBERT vs VADER Scores', fontweight='bold')
    axes[0].axvline(0, color='#888', linestyle='--', linewidth=0.8, alpha=0.5)
    axes[0].axhline(0, color='#888', linestyle='--', linewidth=0.8, alpha=0.5)
    corr = df[['finbert_compound', 'vader_compound']].corr().iloc[0, 1]
    axes[0].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[0].transAxes, fontsize=10, va='top')

    # Distribution
    safe_distribution_plot(df['finbert_compound'], axes[1], '#2196F3', 'FinBERT')
    safe_distribution_plot(df['vader_compound'], axes[1], '#FF9800', 'VADER')
    axes[1].axvline(0, color='#888', linestyle='--', linewidth=0.8, alpha=0.5)
    axes[1].set_title('Sentiment Score Distributions', fontweight='bold')
    axes[1].set_xlabel('Compound Score'); axes[1].legend()

    # Mean by outcome
    comparison = df.groupby('price_up_7d')[['finbert_compound', 'vader_compound']].mean()
    comparison.index = ['DOWN', 'UP']
    comparison.plot(kind='bar', ax=axes[2], color=['#2196F3', '#FF9800'],
                    edgecolor='white', alpha=0.85)
    axes[2].set_title('Mean Sentiment by Stock Outcome', fontweight='bold')
    axes[2].set_ylabel('Mean Compound Score')
    axes[2].set_xlabel('7-Day Outcome')
    axes[2].tick_params(axis='x', rotation=0)
    axes[2].legend(['FinBERT', 'VADER'])

    plt.tight_layout()
    plt.savefig('../outputs/results/06_finbert_vs_vader.png', bbox_inches='tight')
    plt.close()
    print('Saved: 06_finbert_vs_vader.png')
else:
    print('Sentiment columns not found — skipping sentiment comparison plot.')

Saved: 06_finbert_vs_vader.png


## Summary Table

In [9]:
saved = [f for f in os.listdir('../outputs/results') if f.endswith('.png')]
print('=== Final Results Summary ===')
print(f'Best model: {metadata["best_model"]}')
print(f'\nAll model metrics:')
print(metrics_df.round(4).to_string())
print(f'\nSaved {len(saved)} plots to outputs/results/:')
for f in sorted(saved):
    print(f'  {f}')

=== Final Results Summary ===
Best model: XGBoost

All model metrics:
                     accuracy  precision  recall      f1   auroc
model                                                           
Logistic Regression    0.5333     0.5455  0.7500  0.6316  0.4431
Random Forest          0.5000     0.5263  0.6250  0.5714  0.5525
XGBoost                0.5333     0.5625  0.5625  0.5625  0.5725

Saved 8 plots to outputs/results/:
  01_model_comparison.png
  02_roc_curves.png
  03_confusion_matrix.png
  04_feature_importance.png
  04_shap_beeswarm.png
  04b_shap_bar.png
  05_rss_vs_movement.png
  06_finbert_vs_vader.png
